**Melbourne Housing Dataset 데이터셋**

호주 멜버른 지역의 주택 거래 정보를 담고 있는 실제 부동산 데이터입니다. 이 데이터는 주택 판매 기록을 포함하고 있으며, 면적, 방 개수, 건축 연도, 위치 정보 등 다양한 요인을 바탕으로 주택 가격을 분석할 수 있습니다.

제공된 학습용 데이터(mhousing_train.csv)를 이용하여 판매된 가격(Price)를 예측하는 모델을 개발하고, 개발한 모델에 기반하여 평가용 데이터(mhousing_test.csv)에 적용하여 얻은 판매된 가격 예측 값을 아래 [제출형식]에 따라 csv 파일로 생성하여 제출하시오.
- 예측 결과는 RMSLE(Root Mean Squared Log Error) 평가지표에 따라 평가함
- 성능이 우수한 예측 모델을 구축하기 위해서는 데이터 정제, Feature Engineering, 하이퍼 파라미터(hyper parameter) 최적화, 모델 비교 등이 필요할 수 있음. 다만, 과적합에 유의하여야 함


[[제출 형식]]
- 가. CSV 파일명: result.csv(파일명에 디렉토리/폴더 지정불가
- 나. 예측 칼럼명 : pred
- 다. 제출 칼럼 개수 : pred 칼럼 1개
- 라. 평가용 데이터 개수와 예측 결과 데이터 개수 일치 : 1,859개

[[제공 데이터]]
- 데이터 목록
- mhousing_train.csv : 학습용 데이터, 4,337개
- mhousing_test.csv : 평가용 데이터, 1,859개
- 평가용 데이터는 'Price' 칼럼 미제공

In [19]:
# 라이브러리
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, GradientBoostingRegressor
from sklearn.metrics import root_mean_squared_log_error as RMSLE
pd.set_option('display.width', 120)
pd.options.display.float_format = '{:.3f}'.format

# 데이터 불러오기
path = "https://raw.githubusercontent.com/Soyoung-Yoon/bigdata/main/"
train = pd.read_csv(path + "mhousing_train.csv")
test = pd.read_csv(path + "mhousing_test.csv")
# print(train.head(3), train.shape, sep="\n")
# print(test.head(3), test.shape, sep="\n")

# 데이터 전처리
X = train.drop(columns=['Price'])
Y = train['Price']
X_all = pd.concat([X, test])
X_all['Date'] = X_all['Date'].astype('datetime64[ns]')
# print(X_all.head(3))
X_all['Year'] = X_all['Date'].dt.year
X_all['Month'] = X_all['Date'].dt.month
X_all['Day'] = X_all['Date'].dt.day
X_all['Weekday'] = X_all['Date'].dt.day_name()
X_all = X_all.drop(columns=['Date'])
cols_obj = X_all.select_dtypes(include='object').columns
for col in cols_obj:
    X_all[col] = LabelEncoder().fit_transform(X_all[col])
X_all = pd.get_dummies(X_all, drop_first=True, dtype='int')
# print(X_all.head(3))

# 데이터 분할
X = X_all.iloc[:len(X), :]
X_submission = X_all.iloc[len(X):, :]
# print(X.shape, X_submission.shape) # (4337, 21) (1859, 21)
temp = train_test_split(X, Y, test_size=0.3, random_state=123)
x_train, x_test, y_train, y_test = temp
# print(x_train.shape, x_test.shape, y_train.shape, y_test.shape) # (3035, 21) (1302, 21) (3035,) (1302,)

# 파이프라인 모델사전
models = {
    "Linear": Pipeline([
        ('scaler', MinMaxScaler()), ('model', LinearRegression())
    ]),
    "DecisionTree": Pipeline([
        ('model', DecisionTreeRegressor(max_depth=7, random_state=123))
    ]),
    "RandomForest": Pipeline([
        ('model', RandomForestRegressor(max_depth=7, random_state=123))
    ]),
    "AdaBoost": Pipeline([
        ('model', AdaBoostRegressor(n_estimators=900, random_state=123))
    ]),
    "GradientBoosting": Pipeline([
        ('model', GradientBoostingRegressor(n_estimators=900, random_state=123))
    ])
}

# 성능평가 함수
def get_scores(model, x_traiin, x_test, y_train, y_test):
    model.fit(x_train, y_train)
    y_pred1 = model.predict(x_train)
    y_pred2 = model.predict(x_test)
    y_pred1 = np.where(y_pred1 < 0, -y_pred1, y_pred1)
    y_pred2 = np.where(y_pred2 < 0, -y_pred2, y_pred2)
    RMSLE_train = RMSLE(y_train, y_pred1)
    RMSLE_test = RMSLE(y_test, y_pred2)
    return model, RMSLE_train, RMSLE_test

# 모델별 성능평가
results = []
for name, model in models.items():
    model, RMSLE_train, RMSLE_test = get_scores(model, x_train, x_test, y_train, y_test)
    results.append({
        "Model": name, "RMSLE_train": round(RMSLE_train,4), "RMSLE_test": round(RMSLE_test, 4)
    })
res = pd.DataFrame(results).sort_values("RMSLE_test", ascending=True).reset_index(drop=True)
# print(res)

# 모델적합, 예측결과 생성
model = models[res.loc[0, 'Model']]
y_pred = model.predict(X_submission)

# 제출파일 생성
pd.DataFrame({'pred': y_pred}).to_csv("result_type2_3th(2기).csv")

# 결과확인
temp = pd.read_csv("result_type2_3th(2기).csv")
print(temp['pred'].describe())
print("=" * 35)
print(Y[:len(X_submission)].describe())

count      1859.000
mean    1041571.171
std      577158.432
min      162811.511
25%      631045.134
50%      905911.067
75%     1300874.617
max     5425602.203
Name: pred, dtype: float64
count      1859.000
mean    1070975.505
std      665667.428
min      145000.000
25%      615000.000
50%      886000.000
75%     1347500.000
max     5700000.000
Name: Price, dtype: float64
